# 🚀 将 ADK Agent 部署到 Vertex AI Agent Engine

**欢迎来到 Kaggle 5 天 Agents 课程的最后一天！**

在上一份 Notebook 中，你学习了如何使用 Agent2Agent Protocol 让 Agent 之间实现互操作。

现在我们迈出最后一步：使用 [Vertex AI Agent Engine](https://docs.cloud.google.com/agent-builder/agent-engine/overview) 将 Agent 部署到生产环境。

## 💡 扩展你的 Agent

你已经构建了一个很棒的 AI Agent。它在你的本地运行完美：可以聊天、响应智能、看起来一切就绪。但还有一个问题。

> **你的 Agent 还没有公开可用！**

它只存在于 Notebook 和开发环境里。你一旦停止 Notebook 会话，它也会停止工作。你的队友无法访问它，用户也无法与它交互。这正是我们必须部署 Agent 的原因！

## 🎯 你将学到什么

在本 Notebook 中，你将：

- ✅ 构建一个可用于生产环境的 ADK Agent
- ✅ 使用 ADK CLI 将 Agent 部署到 [**Vertex AI Agent Engine**](https://docs.cloud.google.com/agent-builder/agent-engine/overview)
- ✅ 用 Python SDK 测试已部署 Agent
- ✅ 在 Google Cloud Console 中监控和管理已部署 Agent
- ✅ 理解如何通过 Vertex AI Memory Bank 为 Agent 增加 Memory
- ✅ 理解成本管理与资源清理最佳实践

## ⚙️ 第 1 部分：环境准备


### 1.1：⚠️ **重要：前置条件**

本 Notebook 需要一个 **Google Cloud 账号**，以便将 Agent 部署到 Vertex AI Agent Engine。

**如果你还没有 GCP 账号：**

 ✅ 步骤 1：**创建免费的 Google Cloud 账号** - [点此注册](https://cloud.google.com/free)
- 新用户可获得 **$300 免费额度**，有效期 90 天
- 免费试用期间不会产生扣费

 ✅ 步骤 2：**为账号启用 Billing** - 即使免费试用也必须开启
- 需要信用卡进行身份验证
- 只有在你主动升级后才会产生费用
- 只要及时清理资源，本演示通常可保持在 Agent Engine 免费层范围内


 ✅ 步骤 3：**了解免费试用范围** - 明确包含内容
- 查看 [Google Cloud 免费试用详情](https://cloud.google.com/free/docs/free-cloud-features#free-trial)
- 查看 [Google Cloud 免费试用常见问题](https://cloud.google.com/signup-faqs?hl=en#google-cloud-free-trial-faqs)

**💡 快速上手：** 可观看这个 [3 分钟配置视频](https://youtu.be/-nUAQq_evxc)

### 1.2：导入组件

现在导入本 Notebook 需要的组件。这样可以让代码结构更清晰，并确保我们具备所需构建块。

In [6]:
import os
import random
import time
import vertexai
from kaggle_secrets import UserSecretsClient
from vertexai import agent_engines

print("✅ Imports completed successfully")

✅ Imports completed successfully


### 1.3：将 Cloud 凭据添加到 Secrets

1. 在 Notebook 编辑器顶部菜单中选择 `Add-ons`，然后点击 `Google Cloud SDK`。
2. 点击 `Link Account`
3. 选择你的 Google Cloud 账号
4. 将账号附加到 Notebook
   
这个单元会从 Kaggle Secrets 中读取你的 Google Cloud 凭据并完成配置。这样 Notebook 才能鉴权访问 Vertex AI Agent Engine 等 Google Cloud 服务。

In [7]:
# Set up Cloud Credentials in Kaggle
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)

print("✅ Cloud credentials configured")

✅ Cloud credentials configured


### 1.4：设置你的 PROJECT_ID

**重要：** 请将 `"your-project-id"` 替换为你真实的 Google Cloud project ID。你可以在 [Google Cloud Console](https://console.cloud.google.com/) 中查看 project ID。

In [8]:
## Set your PROJECT_ID
PROJECT_ID = "566007917570"  # TODO: Replace with your project ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

if PROJECT_ID == "your-project-id" or not PROJECT_ID:
    raise ValueError("⚠️ Please replace 'your-project-id' with your actual Google Cloud Project ID.")

print(f"✅ Project ID set to: {PROJECT_ID}")

✅ Project ID set to: 566007917570


### 1.5：启用 Google Cloud APIs

在本教程中，你需要在 Google Cloud Console 中启用以下 API：

- Vertex AI API
- Cloud Storage API
- Cloud Logging API
- Cloud Monitoring API
- Cloud Trace API
- Telemetry API

你可以 [点击此链接打开 Google Cloud Console](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com,storage.googleapis.com,logging.googleapis.com,monitoring.googleapis.com,cloudtrace.googleapis.com,telemetry.googleapis.com)，然后按页面指引启用这些 API。

---

## 🏗️ 第 2 部分：使用 ADK 创建你的 Agent

在部署前，我们需要先有一个可用的 Agent。本节会构建一个 **天气助手（Weather Assistant）** 作为示例 Agent。

这个 Agent 为生产测试做了如下配置：

- **Model：** 使用 gemini-2.5-flash-lite，兼顾低延迟与成本效率。
- **Tools：** 包含 `get_weather` 函数，演示工具调用执行。
- **Persona：** 使用对话式回复，体现 instruction-following 能力。

这展示了我们即将打包部署的 ADK 基础架构：**Agent + Tools + Instructions**。

我们将创建如下文件与目录结构：

```
sample_agent/
├── agent.py                  # 逻辑代码
├── requirements.txt          # 依赖库
├── .env                      # secrets/config
└── .agent_engine_config.json # 资源规格
```

### 2.1：创建 Agent 目录

我们需要一个干净的工作目录来打包部署 Agent。这里会创建一个名为 `sample_agent` 的目录。

所有必要文件（包括 Agent 代码、依赖和配置）都会写入该目录，供后续 `adk deploy` 命令使用。

In [9]:
## Create simple agent - all code for the agent will live in this directory
!mkdir -p sample_agent

print(f"✅ Sample Agent directory created")

✅ Sample Agent directory created


### 2.2：创建 requirements 文件

Agent Engine 会为你的 Agent 构建独立运行环境。为确保其正确运行，我们需要声明依赖。

接下来会写入 `requirements.txt`，包含该 Agent 所需的 Python 包。

In [10]:
%%writefile sample_agent/requirements.txt

google-adk
opentelemetry-instrumentation-google-genai

Writing sample_agent/requirements.txt


### 2.3：创建环境配置

我们需要为 Agent 提供必要的云配置。

接下来会写入 `.env` 文件：将 cloud location 设为 `global`，并显式启用 ADK SDK 的 Vertex AI 后端。

In [11]:
%%writefile sample_agent/.env

# https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations#global-endpoint
GOOGLE_CLOUD_LOCATION="global"

# Set to 1 to use Vertex AI, or 0 to use Google AI Studio
GOOGLE_GENAI_USE_VERTEXAI=1

Writing sample_agent/.env


**配置说明：**

- `GOOGLE_CLOUD_LOCATION="global"` - Gemini API 调用使用 `global` endpoint
- `GOOGLE_GENAI_USE_VERTEXAI=1` - 将 ADK 配置为使用 Vertex AI，而不是 Google AI Studio

### 2.4：创建 Agent 代码

现在我们来生成 `agent.py` 文件。这个脚本定义了 **Weather Assistant** 的行为。

Agent 配置：

- 🧠 Model：使用 `gemini-2.5-flash-lite`，兼顾低延迟与成本效率。
- 🛠️ Tools：通过 `get_weather` 函数获取天气数据。
- 📝 Instructions：通过 system prompt 识别城市并用友好语气回复。

In [12]:
%%writefile sample_agent/agent.py
from google.adk.agents import Agent
import vertexai
import os

vertexai.init(
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ["GOOGLE_CLOUD_LOCATION"],
)

def get_weather(city: str) -> dict:
    """
    Returns weather information for a given city.

    This is a TOOL that the agent can call when users ask about weather.
    In production, this would call a real weather API (e.g., OpenWeatherMap).
    For this demo, we use mock data.

    Args:
        city: Name of the city (e.g., "Tokyo", "New York")

    Returns:
        dict: Dictionary with status and weather report or error message
    """
    # Mock weather database with structured responses
    weather_data = {
        "san francisco": {"status": "success", "report": "The weather in San Francisco is sunny with a temperature of 72°F (22°C)."},
        "new york": {"status": "success", "report": "The weather in New York is cloudy with a temperature of 65°F (18°C)."},
        "london": {"status": "success", "report": "The weather in London is rainy with a temperature of 58°F (14°C)."},
        "tokyo": {"status": "success", "report": "The weather in Tokyo is clear with a temperature of 70°F (21°C)."},
        "paris": {"status": "success", "report": "The weather in Paris is partly cloudy with a temperature of 68°F (20°C)."}
    }

    city_lower = city.lower()
    if city_lower in weather_data:
        return weather_data[city_lower]
    else:
        available_cities = ", ".join([c.title() for c in weather_data.keys()])
        return {
            "status": "error",
            "error_message": f"Weather information for '{city}' is not available. Try: {available_cities}"
        }

root_agent = Agent(
    name="weather_assistant",
    model="gemini-2.5-flash-lite",  # Fast, cost-effective Gemini model
    description="A helpful weather assistant that provides weather information for cities.",
    instruction="""
    You are a friendly weather assistant. When users ask about the weather:

    1. Identify the city name from their question
    2. Use the get_weather tool to fetch current weather information
    3. Respond in a friendly, conversational tone
    4. If the city isn't available, suggest one of the available cities

    Be helpful and concise in your responses.
    """,
    tools=[get_weather]
)

Writing sample_agent/agent.py


---

## ☁️ 第 3 部分：部署到 Agent Engine

ADK 支持多种部署平台，详见 [ADK 部署文档](https://google.github.io/adk-docs/deploy/)。

在本 Notebook 中，你将部署到 [Vertex AI Agent Engine](https://docs.cloud.google.com/agent-builder/agent-engine/overview)。

### 🔷 Vertex AI Agent Engine

- **Fully managed**：专为 AI Agent 设计的全托管服务
- **Auto-scaling**：内置自动扩缩容与 session 管理
- **Easy deployment**：可结合 [Agent Starter Pack](https://github.com/GoogleCloudPlatform/agent-starter-pack) 快速部署
- 📚 [部署到 Agent Engine 指南](https://google.github.io/adk-docs/deploy/agent-engine/)

**说明：** 为便于上手，Agent Engine 提供月度免费层，详情见 [文档](https://docs.cloud.google.com/agent-builder/agent-engine/overview#pricing)。如果及时清理资源，本 Notebook 中部署的 Agent 通常可保持在免费层范围内；若长期运行，仍可能产生费用。

### 🚢 其他部署选项

### 🔷 Cloud Run

- Serverless，上手最简单
- 适合演示与中小规模负载
- 📚 [部署到 Cloud Run 指南](https://google.github.io/adk-docs/deploy/cloud-run/)

### 🔷 Google Kubernetes Engine (GKE)

- 对容器化部署拥有完整控制能力
- 适合复杂多智能体系统
- 📚 [部署到 GKE 指南](https://google.github.io/adk-docs/deploy/gke/)

### 3.1：创建部署配置

`.agent_engine_config.json` 文件用于控制部署参数设置。

In [13]:
%%writefile sample_agent/.agent_engine_config.json
{
    "min_instances": 0,
    "max_instances": 1,
    "resource_limits": {"cpu": "1", "memory": "1Gi"}
}

Writing sample_agent/.agent_engine_config.json


**配置说明：**

- `"min_instances": 0` - 空闲时缩容到 0（节省成本）
- `"max_instances": 1` - 最多仅运行 1 个实例（本示例足够）
- `"cpu": "1"` - 每个实例分配 1 个 CPU 核心
- `"memory": "1Gi"` - 每个实例分配 1 GB 内存

这些参数在保证天气 Agent 运行所需资源的同时，尽量将成本控制在较低水平。

### 3.2：选择部署区域

Agent Engine 仅在部分区域可用。本示例会随机选择一个可用区域。

In [14]:
regions_list = ["europe-west1", "europe-west4", "us-east4", "us-west1"]
deployed_region = random.choice(regions_list)

print(f"✅ Selected deployment region: {deployed_region}")

✅ Selected deployment region: us-west1


**关于区域（Region）：**

Agent Engine 支持多个区域。用于生产时建议：

- 选择靠近用户的区域以降低延迟
- 考虑数据驻留（data residency）要求
- 查阅 [Agent Engine 区域文档](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview#locations)

### 3.3：部署 Agent

这里会使用 ADK CLI 将你的 Agent 部署到 Agent Engine。

In [17]:
!adk deploy agent_engine --project=$PROJECT_ID --region=$deployed_region sample_agent --agent_engine_config_file=sample_agent/.agent_engine_config.json

Staging all files in: /kaggle/working/sample_agent_tmp20260317_141610
Copying agent source code...
Copying agent source code complete.
Resolving files and dependencies...
Reading agent engine config from sample_agent/.agent_engine_config.json
Reading environment variables from /kaggle/working/sample_agent/.env
Ignoring GOOGLE_CLOUD_LOCATION in .env as `--region` was explicitly passed and takes precedence
Initializing Vertex AI...
Vertex AI initialized.
Created sample_agent_tmp20260317_141610/agent_engine_app.py
Files and dependencies resolved
Deploying to agent engine...
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of source_packages
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=566007917570.
INFO:vertexai_genai.agentengines:Agent Engine created. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines

**刚才发生了什么：**

`adk deploy agent_engine` 命令会：

1. 打包你的 Agent 代码（`sample_agent/` 目录）
2. 上传到 Agent Engine
3. 创建容器化部署
4. 输出类似这样的资源名：`projects/PROJECT_NUMBER/locations/REGION/reasoningEngines/ID`

**说明：** 部署通常需要 2-5 分钟。

---

## 🤖 第 4 部分：获取并测试已部署 Agent

### 4.1：获取已部署 Agent

通过 CLI 完成部署后，我们需要获取 Agent 对象，才能与它交互。

In [19]:
# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=deployed_region)

# Get the most recently deployed agent
agents_list = list(agent_engines.list())
if agents_list:
    remote_agent = agents_list[0]  # Get the first (most recent) agent
    client = agent_engines
    print(f"✅ Connected to deployed agent: {remote_agent.resource_name}")
else:
    print("❌ No agents found. Please deploy first.")

✅ Connected to deployed agent: projects/566007917570/locations/us-west1/reasoningEngines/4944020004992450560


**刚才发生了什么：**

这个单元完成了已部署 Agent 的获取：

1. 使用你的 project 和 region 初始化 Vertex AI SDK
2. 列出该区域内所有已部署 Agent
3. 获取第一个（最近部署的）Agent
4. 将其保存为 `remote_agent` 用于后续测试

### 4.2：测试已部署 Agent

现在我们向已部署 Agent 发送一个查询！

In [20]:
async for item in remote_agent.async_stream_query(
    message="What is the weather in Tokyo?",
    user_id="user_42",
):
    print(item)

{'model_version': 'gemini-2.5-flash-lite', 'content': {'parts': [{'function_call': {'id': 'adk-3c8589c6-7a65-4c54-9e36-a4b4188a1bec', 'args': {'city': 'Tokyo'}, 'name': 'get_weather'}}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 5, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 5}], 'prompt_token_count': 232, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 232}], 'total_token_count': 237, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.4200003147125244, 'invocation_id': 'e-a76d0049-e63d-46ef-98cd-b24162349c4a', 'author': 'weather_assistant', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'long_running_tool_ids': [], 'id': '9d05e56d-6f5f-4442-a4ea-5cef8c6b6834', 'timestamp': 1773757595.702913}
{'content': {'parts': [{'function_response': {'id': 'adk-3c8589c6-7a65-4c54-9e36-a4b4188a1bec', 'name': 'get_weather', 'response': {'status': 'su

**刚才发生了什么：**

这个单元用于测试你已部署的 Agent：

1. 发送查询："What is the weather in Tokyo?"
2. 流式接收 Agent 的响应

**如何理解输出：**

你会看到多个输出项：

1. **Function call** - Agent 决定调用 `get_weather` 工具
2. **Function response** - 工具返回结果（天气数据）
3. **Final response** - Agent 的自然语言回答

---

## 🧠 第 5 部分：使用 Vertex AI Memory Bank 实现长期记忆

### Memory Bank 解决了什么问题？

你部署的 Agent 具备 **session memory**：在当前对话期间它能记住上下文。但会话结束后，它会遗忘一切；每次新会话都从头开始。

**典型问题：**

- 用户今天告诉 Agent：「我偏好摄氏度（Celsius）」
- 明天再问天气时，Agent 又用华氏度（Fahrenheit）回复（忘了偏好）
- 用户不得不反复重复偏好

### 💡 什么是 Vertex AI Memory Bank？

Memory Bank 为 Agent 提供 **跨会话的长期记忆**：

| Session Memory | Memory Bank |
|---------------|-------------|
| 单次对话 | 所有历史对话 |
| 会话结束即遗忘 | 可长期保留 |
| “我刚刚说了什么？” | “我最喜欢哪个城市？” |

**工作机制：**

1. **对话进行中** - Agent 通过 memory tools 检索历史事实
2. **对话结束后** - Agent Engine 提取关键信息（如“用户偏好摄氏度”）
3. **下次会话** - Agent 自动回忆并使用这些信息

**示例：**

- **Session 1：** 用户：“我偏好摄氏度”
- **Session 2（几天后）：** 用户：“东京天气？” → Agent 会自动用摄氏度回复 ✨

### 🔧 Memory Bank 与当前部署的关系

你的 Agent Engine 部署**已经提供 Memory Bank 的基础设施**，但默认并未开启。

**要使用 Memory Bank：**

1. 在 Agent 代码中添加 memory tools（`PreloadMemoryTool`）
2. 添加 callback，将对话保存到 Memory Bank
3. 重新部署 Agent

配置完成后，Memory Bank 会自动工作，无需额外基础设施。

### 📚 延伸阅读

- **[ADK Memory 指南](https://google.github.io/adk-docs/sessions/memory/)** - 含完整说明与代码示例
- **[Memory Tools](https://google.github.io/adk-docs/tools/built-in-tools/)** - PreloadMemory 与 LoadMemory 文档
- **[Get started with Memory Bank on ADK](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/memory_bank/get_started_with_memory_bank_on_adk.ipynb)** - 演示如何构建带 memory 能力的 ADK Agent

---

## 🧹 第 6 部分：资源清理

**⚠️ 重要：为避免意外扣费，测试完成后请务必删除资源！**

**成本提醒**

再次提醒：如果 Agent 持续运行，可能会产生费用。Agent Engine 提供月度免费层，详情见 [文档](https://docs.cloud.google.com/agent-builder/agent-engine/overview#pricing)。

**测试完成后请务必清理资源！**

当你完成对已部署 Agent 的测试与调用后，建议删除远程 Agent 以避免额外费用：

In [24]:
agent_engines.delete(resource_name=remote_agent.resource_name, force=True)

print("✅ Agent successfully deleted")

✅ Agent successfully deleted


**刚才发生了什么：**

这个单元会删除你已部署的 Agent：

- `resource_name=remote_agent.resource_name` - 指定要删除的 Agent
- `force=True` - 即使 Agent 正在运行也强制删除

删除过程通常需要 1-2 分钟。你可以在 [Agent Engine Console](https://console.cloud.google.com/vertex-ai/agents/agent-engines) 中确认删除状态。

---

## ✅ 恭喜！你已具备生产部署能力

你已经成功掌握了如何将 ADK Agent 部署到 Vertex AI Agent Engine，把 Agent 从开发环境带到生产环境！

现在你已经知道如何依托企业级基础设施部署 Agent、管理成本并验证生产部署效果。

### 📚 延伸阅读

可参考以下文档继续深入：

- [ADK 文档](https://google.github.io/adk-docs/)
- [Agent Engine 文档](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview)
- [ADK 部署指南](https://google.github.io/adk-docs/deploy/agent-engine/)

**其他部署选项：**

- [Cloud Run 部署](https://google.github.io/adk-docs/deploy/cloud-run/)
- [GKE 部署](https://google.github.io/adk-docs/deploy/gke/)

**生产最佳实践：**

- 测试完成后删除部署实例，避免不必要费用
- 为调试启用 tracing（`enable_tracing=True`）
- 通过 [Vertex AI Console](https://console.cloud.google.com/vertex-ai/agents/agent-engines) 进行监控
- 遵循 [安全最佳实践](https://google.github.io/adk-docs/safety/)

## 🎯 课程回顾：你的 5 天学习之旅

过去 5 天，你已经学习了：

- **Day 1：** Agent 基础 - 使用工具与指令构建第一个 Agent
- **Day 2：** 高级工具 - 自定义工具、内置工具与最佳实践
- **Day 3：** Sessions 与 Memory - 管理对话与长期知识
- **Day 4：** 可观测性与评估 - 监控 Agent 并衡量表现
- **Day 5：** 生产部署 - 使用 Agent Engine 让 Agent 正式上线

你现在已经具备构建、测试和部署生产级 AI Agent 的完整工具链！

### 🚀 接下来做什么？

**感谢你完成 5 天 AI Agents 课程！**

现在轮到你动手了：
- 开始用 ADK 构建你自己的 AI Agent
- 在 [Kaggle Discord](https://discord.com/invite/kaggle) 与社区分享你的项目
- 在 [ADK 文档](https://google.github.io/adk-docs/) 中探索更高级模式

**Happy building! 🚀** 